In [ ]:
# | default_exp wuilt.optimizer

# Wuilt Optimizer
> Generate optimized SEO fields for Wuilt products using DSPy.

In [ ]:
# | export
import os, dspy
from seootter.wuilt.client import WuiltClient
from seootter.dspy_infer import configure_lm

In [ ]:
# | export

class GenerateProductSEO(dspy.Signature):
    """You are an e-commerce SEO expert. Generate optimized SEO fields for a product based on its current content."""
    title: str = dspy.InputField(desc="Current product title")
    description_html: str = dspy.InputField(desc="Current product description HTML")
    short_description: str = dspy.InputField(desc="Current short product description")
    language: str = dspy.InputField(desc="Language code e.g. 'en', 'ar'")
    seo_title: str = dspy.OutputField(desc="Optimized SEO title (50-60 characters, include focus keyword, brand name at end)")
    seo_description: str = dspy.OutputField(desc="Optimized meta description (150-160 characters, compelling with CTA, include keyword)")
    optimized_description_html: str = dspy.OutputField(desc="Improved product description HTML — benefit-driven, scannable, rich formatting, include keywords naturally")
    optimized_short_description: str = dspy.OutputField(desc="Concise value proposition (2-3 sentences, highlight unique benefit)")

In [ ]:
# | export

class GenerateStoreSEO(dspy.Signature):
    """You are an e-commerce SEO expert. Generate optimized SEO fields for a store."""
    store_name: str = dspy.InputField(desc="Current store name")
    language: str = dspy.InputField(desc="Language code e.g. 'en', 'ar'")
    seo_title: str = dspy.OutputField(desc="Optimized store-level SEO title (50-60 chars, include brand)")
    seo_description: str = dspy.OutputField(desc="Optimized store-level meta description (150-160 chars)")


In [ ]:
# | export

class GeneratePageSEO(dspy.Signature):
    """You are an e-commerce SEO expert. Generate optimized SEO fields for a store page."""
    page_name: str = dspy.InputField(desc="Current page name")
    page_type: str = dspy.InputField(desc="Page type: HOME or CUSTOM")
    language: str = dspy.InputField(desc="Language code e.g. 'en', 'ar'")
    seo_title: str = dspy.OutputField(desc="Optimized page SEO title (50-60 chars)")
    seo_description: str = dspy.OutputField(desc="Optimized page meta description (150-160 chars)")


In [ ]:
# | export

class GenerateRobotsTxt(dspy.Signature):
    """You are an SEO expert. Generate an optimized robots.txt for an e-commerce store."""
    store_name: str = dspy.InputField(desc="Store name")
    store_domain: str = dspy.InputField(desc="Store domain")
    language: str = dspy.InputField(desc="Language code e.g. 'en', 'ar'")
    robots_content: str = dspy.OutputField(desc="Complete robots.txt content with User-agent, Disallow, Allow, Sitemap directives")


In [ ]:
# | export

def optimize_product(
    product_title: str,
    description_html: str = "",
    short_description: str = "",
    language: str = "en",
) -> dict:
    "Generate optimized SEO fields for a single product using DSPy."
    configure_lm()
    predictor = dspy.Predict(GenerateProductSEO)
    truncated_desc = description_html[:2000]
    truncated_short = short_description[:500]
    result = predictor(
        title=product_title[:200],
        description_html=truncated_desc,
        short_description=truncated_short,
        language=language,
    )
    return {
        "seo_title": result.seo_title,
        "seo_description": result.seo_description,
        "optimized_description_html": result.optimized_description_html,
        "optimized_short_description": result.optimized_short_description,
    }

In [ ]:
# | export

def optimize_and_push(
    client: WuiltClient,
    product_id: str,
    product_title: str,
    description_html: str = "",
    short_description: str = "",
    language: str = "en",
) -> dict:
    "Generate optimized SEO for a product and push it back to Wuilt."
    optimized = optimize_product(
        product_title=product_title,
        description_html=description_html,
        short_description=short_description,
        language=language,
    )
    result = client.update_product_seo(
        product_id=product_id,
        seo_title=optimized["seo_title"],
        seo_description=optimized["seo_description"],
        description_html=optimized["optimized_description_html"],
        short_description=optimized["optimized_short_description"],
    )
    return {
        "optimized": optimized,
        "push_result": result,
    }

In [ ]:
# | export

def batch_optimize_products(
    api_key: str,
    store_id: str,
    locale: str = "ar",
    product_ids: list[str] | None = None,
) -> list[dict]:
    "Optimize SEO for all (or specified) products in a store and push to Wuilt."
    client = WuiltClient(api_key=api_key, store_id=store_id, locale=locale)
    products = client.get_products()

    if product_ids:
        products = [p for p in products if p["id"] in product_ids]

    results = []
    for p in products:
        print(f"Optimizing: {p['title']} ({p['id']})")
        seo = p.get("seo") or {}
        try:
            result = optimize_and_push(
                client=client,
                product_id=p["id"],
                product_title=p.get("title", ""),
                description_html=p.get("descriptionHtml", ""),
                short_description=p.get("shortDescription", ""),
                language=locale,
            )
            result["product_id"] = p["id"]
            result["title"] = p["title"]
            result["old_seo"] = seo
            results.append(result)
            print(f"  -> SEO title: {result['optimized']['seo_title'][:60]}...")
        except Exception as e:
            print(f"  FAILED: {e}")
            results.append({"product_id": p["id"], "title": p["title"], "error": str(e)})

    return results

In [ ]:
# | hide
# | eval: false
from dotenv import load_dotenv
import os

load_dotenv()

result = optimize_product(
    product_title="RED REX Beef Mass Plus",
    description_html="<p>Premium beef protein mass gainer for serious athletes. Packed with 50g protein per serving.</p>",
    short_description="Premium beef mass gainer, 50g protein",
    language="en",
)
print(f"SEO Title: {result['seo_title']}")
print(f"SEO Desc: {result['seo_description']}")
print(f"Optimized Desc HTML: {result['optimized_description_html']}")
print(f"Optimized Short Desc: {result['optimized_short_description']}")

SEO Title: RED REX Beef Mass Plus - High-Protein Gainer by RED REX
SEO Desc: Boost muscle growth with RED REX Beef Mass Plus, a premium beef protein mass gainer. Get 50g protein per serving and unlock your athletic potential - Shop now and fuel your gains!
Optimized Desc HTML: <p>Unlock your full athletic potential with RED REX Beef Mass Plus, a premium <b>beef protein mass gainer</b> designed for serious athletes. Each serving is packed with <b>50g of protein</b> to support muscle growth and recovery. Whether you're looking to bulk up or maintain your current physique, our high-protein formula helps you achieve your goals. <b>Key benefits</b> include:</p>
<ul>
  <li>High-quality beef protein for enhanced muscle growth</li>
  <li>50g protein per serving to support muscle recovery</li>
  <li>Supports serious athletes in their training and competition</li>
</ul>
<p>Choose RED REX Beef Mass Plus for a <b>premium beef protein mass gainer</b> that delivers results. Fuel your gains and take 